# Part B Q1: Dataset and Exploratory Data Analysis

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import chi2

from clean_review import clean_review

DATA_DIR = Path("../data")
RAW_FILE = DATA_DIR / "yelp_review_full_raw_30k.csv"
CLEAN_FILE = DATA_DIR / "yelp_clean.csv"
OUT_DIR = Path("eda_outputs")
OUT_DIR.mkdir(exist_ok=True)

TEXT_COL = "review"
LABEL_COL = "rating"
SENT_COL = "sentiment"

## 1. Data Source

In [ ]:
raw = pd.read_csv(RAW_FILE)

print("Rows            :", len(raw))
print("Columns         :", list(raw.columns))
print("Missing values  :", raw.isnull().sum().sum())
print("Reviews per star:")
print(raw[LABEL_COL].value_counts().sort_index().to_string())

balanced = raw[LABEL_COL].value_counts().nunique() == 1
print("\nExactly balanced across the 5 star ratings:", balanced)

## 2. Load Data and Derive the Target

In [ ]:
def to_sentiment(rating):
    return np.where(rating <= 2, "negative", np.where(rating == 3, "neutral", "positive"))


df = raw.copy()
df[SENT_COL] = to_sentiment(df[LABEL_COL])

print("Sample records:")
print(df.head(3).to_string())

## 3. Data Quality

In [ ]:
print("Missing values")
print(df[[TEXT_COL, LABEL_COL]].isnull().sum().to_string())

print("\nExact duplicate review texts:", df.duplicated(subset=[TEXT_COL]).sum())

## 4. Data Preparation and Cleaning

In [ ]:
df = df.dropna(subset=[TEXT_COL, LABEL_COL]).copy()
df["clean_text"] = df[TEXT_COL].apply(clean_review)

before = len(df)
df = df[df["clean_text"].str.strip() != ""].copy()

print("Rows before cleaning:", before)
print("Rows after cleaning :", len(df), " (empty rows dropped:", before - len(df), ")")

## 5. Class Distribution

In [ ]:
order = ["negative", "neutral", "positive"]
raw_dist = to_sentiment(raw[LABEL_COL])
raw_counts = pd.Series(raw_dist).value_counts().reindex(order)
clean_counts = df[SENT_COL].value_counts().reindex(order)

print(pd.DataFrame({"before cleaning": raw_counts, "after cleaning": clean_counts}).to_string())

plt.figure(figsize=(7, 4))
plt.bar(order, clean_counts.values, color=["#C0392B", "#C9B037", "#2E8B57"])
plt.title("Yelp reviews per sentiment class (after cleaning)")
plt.xlabel("Sentiment class")
plt.ylabel("Number of reviews")
plt.tight_layout()
plt.savefig(OUT_DIR / "class_distribution.png", dpi=120)
plt.show()

## 6. Review Length Before and After Cleaning

In [ ]:
df["char_count"] = df[TEXT_COL].astype(str).str.len()
df["words_before"] = df[TEXT_COL].astype(str).str.split().str.len()
df["words_after"] = df["clean_text"].str.split().str.len()

print("Length summary")
print(df[["char_count", "words_before", "words_after"]].describe().round(1).to_string())

print("\nAverage cleaned word count by class")
print(df.groupby(SENT_COL)["words_after"].mean().round(1).reindex(order).to_string())

plt.figure(figsize=(8, 4))
plt.hist(df["words_before"], bins=50, color="#C44E52", alpha=0.7, label="before cleaning")
plt.hist(df["words_after"], bins=50, color="#4C72B0", alpha=0.7, label="after cleaning")
plt.title("Review length distribution (words)")
plt.xlabel("Words per review")
plt.ylabel("Number of reviews")
plt.xlim(0, 400)
plt.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "length_distribution.png", dpi=120)
plt.show()

## 7. Feature Space

In [ ]:
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=2, sublinear_tf=True)
X = vectorizer.fit_transform(df["clean_text"])
features = np.array(vectorizer.get_feature_names_out())

vocabulary = set()
for doc in df["clean_text"]:
    vocabulary.update(doc.split())

density = X.nnz / (X.shape[0] * X.shape[1])

print("Distinct words after cleaning :", len(vocabulary))
print("TF-IDF features kept          :", X.shape[1])
print("Matrix shape                  :", X.shape)
print(f"Non-zero cells                : {X.nnz:,} ({density:.2%} of the matrix)")
print(f"Sparsity                      : {1 - density:.2%} zeros")

## 8. Class-Separating Terms

In [ ]:
def chi2_top_words_per_class(X, features, labels, top_n=12):
    result = {}
    for label in sorted(labels.unique()):
        mask = (labels == label).values
        scores, _ = chi2(X, mask.astype(int))

        # Keep only words that are heavier INSIDE the class than outside it.
        mean_in = np.asarray(X[mask].mean(axis=0)).ravel()
        mean_out = np.asarray(X[~mask].mean(axis=0)).ravel()
        characteristic = mean_in > mean_out

        order = np.argsort(scores)[::-1]
        picks = [i for i in order if characteristic[i]][:top_n]
        result[label] = [(features[i], float(scores[i])) for i in picks]
    return result


chi_words = chi2_top_words_per_class(X, features, df[SENT_COL])

for label in sorted(chi_words):
    print(f"{label}: " + ", ".join(word for word, _ in chi_words[label]))

In [ ]:
spectrum = ["#C0392B", "#C9B037", "#2E8B57"]
fig, axes = plt.subplots(len(chi_words), 1, figsize=(9, 2.6 * len(chi_words)))

for ax, label, color in zip(axes, sorted(chi_words), spectrum):
    pairs = chi_words[label][::-1]   # largest ends up on top
    ax.barh([word for word, _ in pairs], [score for _, score in pairs], color=color)
    ax.set_title(f"{label} - most distinctive words (chi-square)", fontsize=10)
    ax.tick_params(axis="y", labelsize=8)

fig.supxlabel("chi-square score (word vs this sentiment class)")
fig.tight_layout()
fig.savefig(OUT_DIR / "chi2_top_words_per_class.png", dpi=120)
plt.show()

### Class Vocabulary Comparison

## 9. Save the Cleaned Dataset

In [ ]:
out = df[[TEXT_COL, "clean_text", LABEL_COL, SENT_COL]].copy()
out.to_csv(CLEAN_FILE, index=False)

print("Saved  :", CLEAN_FILE)
print("Columns:", list(out.columns))
print("Rows   :", len(out))